# Video → 3D (VGGT on Colab)

**Runtime → Change runtime type → T4 GPU** before running.

If imports fail after install: **Runtime → Restart session**, then run from the install cell again.


In [ ]:
# 1) Install base deps into THIS notebook's Python
import sys
!{sys.executable} -m pip -q install "numpy<2" Pillow huggingface_hub einops safetensors opencv-python-headless trimesh matplotlib scipy tqdm


In [ ]:
# 2) Clone VGGT and make it importable (Colab-safe)
import os, sys, shutil

# Old mistake: a folder named /content/vggt shadows the real package
if os.path.isdir("/content/vggt") and not os.path.isfile("/content/vggt/models/vggt.py"):
    print("Removing shadowing /content/vggt ...")
    shutil.rmtree("/content/vggt")

SRC = "/content/vggt_src"
if not os.path.isdir(os.path.join(SRC, "vggt", "models")):
    !git clone --depth 1 https://github.com/facebookresearch/vggt.git {SRC}

# Install with the same Python the notebook kernel uses
!{sys.executable} -m pip -q install -e {SRC}

# Critical on Colab: put the repo root on sys.path so `import vggt` finds vggt_src/vggt/
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
print("VGGT import OK")
print("sys.path[0] =", sys.path[0])


In [ ]:
# 3) Upload your video
from google.colab import files
uploaded = files.upload()
assert uploaded, "Upload one video file"
VIDEO_PATH = list(uploaded.keys())[0]
print("Using video:", VIDEO_PATH)


In [ ]:
# 4) Sample frames (~1 fps, max 40)
import cv2
from pathlib import Path

frames_dir = Path("/content/frames")
frames_dir.mkdir(exist_ok=True)
for p in frames_dir.glob("*"):
    p.unlink()

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
interval = max(1, int(round(fps)))
max_frames = 40
idx = saved = 0
while saved < max_frames:
    ok, frame = cap.read()
    if not ok:
        break
    if idx % interval == 0:
        cv2.imwrite(str(frames_dir / f"{saved:06d}.jpg"), frame)
        saved += 1
    idx += 1
cap.release()
image_names = sorted(str(p) for p in frames_dir.glob("*.jpg"))
print(f"Extracted {len(image_names)} frames")
assert len(image_names) >= 2, "Need at least 2 frames" 


In [ ]:
# 5) Load model + run reconstruction
import sys, torch
SRC = "/content/vggt_src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "Enable GPU: Runtime → Change runtime type → T4 GPU"

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval()

images = load_and_preprocess_images(image_names).to(device)
print("images:", tuple(images.shape))

dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        predictions = model(images)

for k, v in list(predictions.items()):
    if isinstance(v, torch.Tensor):
        predictions[k] = v.detach().cpu()
print("keys:", sorted(predictions.keys()))


In [ ]:
# 6) Export GLB
import sys
SRC = "/content/vggt_src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from visual_util import predictions_to_glb

scene = predictions_to_glb(
    predictions,
    conf_thres=50.0,
    filter_by_frames="all",
    show_cam=True,
    prediction_mode="Predicted Pointmap",
)
out_path = "/content/scene.glb"
scene.export(out_path)
print("Wrote", out_path)


In [ ]:
# 7) Download GLB for your Streamlit app
from google.colab import files
files.download("/content/scene.glb")
print("Attach this scene.glb to your job in the UI.")
